In [ ]:
# | default_exp cli

In [ ]:
# | export
import typer
import sys
from pathlib import Path
from typing import Optional
import pandas as pd
import structlog

log = structlog.get_logger()
app = typer.Typer(name="kreview", help="ctDNA fragmentomics feature evaluation")

# Register evaluation subcommands (kreview eval cpu|gpu|multimodal)
from kreview.cli_eval import eval_app

app.add_typer(eval_app, name="eval")

# Register select command (kreview select)
from kreview.cli_select import select

app.command(name="select")(select)


def version_callback(value: bool):
    if value:
        import kreview

        typer.echo(f"kreview version: {kreview.__version__}")
        raise typer.Exit()


@app.callback()
def main(
    version: Optional[bool] = typer.Option(
        None, "--version", callback=version_callback, is_eager=True
    ),
):
    """Kreview: ctDNA fragmentomics feature evaluation framework."""
    pass


In [ ]:
# | export
@app.command()
def label(
    cancer_samplesheet: Path = typer.Option(..., help="Cancer samplesheet CSV"),
    healthy_xs1_samplesheet: Path = typer.Option(
        ..., help="Healthy XS1 samplesheet CSV"
    ),
    healthy_xs2_samplesheet: Path = typer.Option(
        ..., help="Healthy XS2 samplesheet CSV"
    ),
    cbioportal_dir: Path = typer.Option(..., help="Directory with cBioPortal files"),
    output: Path = typer.Option("labels.parquet", help="Output parquet file"),
    min_vaf: float = typer.Option(
        0.01, help="Min VAF for Possible ctDNA+ (default 1%)"
    ),
    min_fragments: int = typer.Option(
        2000, help="Min fragments PF for Depth QC (samples below are Insufficient Data)"
    ),
    min_variants: int = typer.Option(
        1, help="Min # variants passing VAF for Possible ctDNA+"
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        help="Optional TSV of CH hotspot variants for CH-only demotion. "
        "Samples with only CH mutations are demoted to Possible ctDNA−.",
    ),
):
    """Generate ctDNA labels without feature evaluation."""
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler

    print("=== kreview label ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --cancer-samplesheet : {cancer_samplesheet}", flush=True)
    print(f"  --healthy-xs1       : {healthy_xs1_samplesheet}", flush=True)
    print(f"  --healthy-xs2       : {healthy_xs2_samplesheet}", flush=True)
    print(f"  --cbioportal-dir    : {cbioportal_dir}", flush=True)
    print(f"  --output            : {output}", flush=True)
    print(f"  --min-vaf           : {min_vaf}", flush=True)
    print(f"  --min-variants      : {min_variants}", flush=True)
    print(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}", flush=True)
    print("", flush=True)

    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        [],
    )
    # Metadata is ~1 row/sample — load everything in a single large batch.
    config = LabelConfig(
        min_vaf=min_vaf,
        min_fragments=min_fragments,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )

    labeler = CtDNALabeler(paths, config)
    labels = labeler.label_all()
    labels.to_parquet(output, index=False)
    typer.echo(f"Labels written to {output}")
    typer.echo(labels["label"].value_counts().to_string())


In [ ]:
# | export
@app.command()
def features_list():
    """List all registered feature evaluators."""
    from kreview.registry import get_all_evaluators

    evals = get_all_evaluators()
    for e in evals:
        typer.echo(
            f"Tier {getattr(e, 'tier', '?')}: {getattr(e, 'name', type(e).__name__)} ({getattr(e, 'source_file', '?')})"
        )


In [ ]:
# | export
import numpy as np
import time


# _impute is now in selection.py — import it for backward compat
from kreview.selection import _impute


def _extract_evaluator(
    evaluator,
    paths,
    all_sample_ids: list[str],
    chunk_size: int,
    out_path: Path,
    labels_df: pd.DataFrame,
    *,
    _echo=print,
) -> pd.DataFrame | None:
    """Extract features for a single evaluator and write the matrix parquet.

    Tries SQL pushdown first (Path A); falls back to chunked Python
    streaming (Path B) when the evaluator doesn't support SQL or SQL
    returns empty results.

    Returns:
        Merged DataFrame (labels + features) written to disk, or None
        if extraction produced no data.
    """
    from kreview.core import iter_feature_chunks, run_feature_sql

    _echo(f"  Extracting '{evaluator.name}'...")
    t1 = time.time()

    # --- Path A: SQL Pushdown (if evaluator supports it) ---
    sql_query = evaluator.extract_sql()
    feat_matrix = None

    if sql_query is not None:
        _echo(f"  Using SQL pushdown for '{evaluator.name}'...")
        sql_df = run_feature_sql(
            sql_query,
            str(evaluator.source_file),
            paths.krewlyzer_dirs,
            set(all_sample_ids),
        )
        if not sql_df.empty:
            feat_matrix = sql_df
            extract_sec = time.time() - t1
            n_samples = (
                sql_df["sample_id"].nunique()
                if "sample_id" in sql_df.columns
                else len(sql_df)
            )
            _echo(
                f"  SQL pushdown complete: {n_samples} samples "
                f"in {extract_sec:.1f}s"
            )
            # Normalize sample_id column name for downstream merge
            if "sample_id" in sql_df.columns and "SAMPLE_ID" not in sql_df.columns:
                feat_matrix = feat_matrix.rename(columns={"sample_id": "SAMPLE_ID"})
        else:
            _echo(
                f"  SQL pushdown returned empty result for '{evaluator.name}', "
                f"falling back to chunked extraction..."
            )
            sql_query = None  # Force fallback to Path B

    # --- Path B: Chunked Python streaming (default fallback) ---
    if sql_query is None:
        _echo(f"  Streaming cohort from DuckDB (chunk_size={chunk_size})...")
        partial_results = []
        failed_samples = []
        total_samples_processed = 0
        total_rows_read = 0

        for chunk_df, chunk_idx, n_chunks in iter_feature_chunks(
            str(evaluator.source_file),
            paths.krewlyzer_dirs,
            set(all_sample_ids),
            chunk_size=chunk_size,
        ):
            sid_col = "sample_id" if "sample_id" in chunk_df.columns else "SAMPLE_ID"
            chunk_sample_ids = chunk_df[sid_col].unique()
            chunk_n_rows = len(chunk_df)
            total_rows_read += chunk_n_rows

            _echo(
                f"  Chunk {chunk_idx + 1}/{n_chunks}: "
                f"{len(chunk_sample_ids)} samples, {chunk_n_rows:,} rows"
            )

            # Extract features from each sample within this chunk
            chunk_extracted = 0
            for sample_id in chunk_sample_ids:
                sample_df = chunk_df[chunk_df[sid_col] == sample_id]
                try:
                    res = evaluator.extract(sample_df)
                    if res:
                        res["SAMPLE_ID"] = sample_id
                        partial_results.append(pd.DataFrame([res]))
                        chunk_extracted += 1
                except Exception as exc:
                    log.warning(
                        "sample_extraction_failed",
                        evaluator=evaluator.name,
                        sample_id=sample_id,
                        error=str(exc),
                    )
                    failed_samples.append(sample_id)

            total_samples_processed += len(chunk_sample_ids)
            _echo(
                f"    Extracted {chunk_extracted}/" f"{len(chunk_sample_ids)} samples"
            )

            # Explicit memory release — reclaim ~5GB before next chunk
            del chunk_df

        if total_samples_processed == 0:
            _echo(f"  WARNING: No data found for {evaluator.name}, skipping")
            return None

        extract_sec = time.time() - t1
        _echo(
            f"  Extraction complete: {total_samples_processed} samples, "
            f"{total_rows_read:,} rows read in {extract_sec:.1f}s"
        )

        if failed_samples:
            failed_path = out_path / f"{evaluator.name}_failed_samples.csv"
            pd.DataFrame({"SAMPLE_ID": failed_samples}).to_csv(failed_path, index=False)
            _echo(
                f"  WARNING: {len(failed_samples)} samples failed extraction. "
                f"Saved to {failed_path.name}"
            )
            log.warning(
                "extraction_failures",
                evaluator=evaluator.name,
                n_failed=len(failed_samples),
                output=str(failed_path),
            )

        if not partial_results:
            _echo(f"  WARNING: No results for {evaluator.name}, skipping")
            return None

        feat_matrix = pd.concat(partial_results, ignore_index=True)

    # Guard: both Path A and Path B should set feat_matrix before reaching
    # this point (or return None). This is a safety net.
    if feat_matrix is None:
        _echo(f"  WARNING: No feature matrix produced for {evaluator.name}, skipping")
        return None

    merged = pd.merge(labels_df, feat_matrix, on="SAMPLE_ID", how="inner")
    out_p = out_path / f"{evaluator.name}_matrix.parquet"
    merged.to_parquet(out_p, index=False)
    _echo(f"  Matrix: {merged.shape[0]} samples x {merged.shape[1]} cols -> {out_p}")

    return merged


def _find_quarto() -> str:
    """Discover the quarto binary from PATH or well-known install locations."""
    import shutil

    # 1. Check PATH first
    found = shutil.which("quarto")
    if found:
        return found

    # 2. Check well-known install locations
    candidates = [
        # macOS Positron bundled Quarto
        "/Applications/Positron.app/Contents/Resources/app/quarto/bin/quarto",
        # macOS standalone Quarto install
        "/Applications/quarto/bin/quarto",
        # Homebrew (Apple Silicon)
        "/opt/homebrew/bin/quarto",
        # Homebrew (Intel)
        "/usr/local/bin/quarto",
        # Linux system install
        "/usr/bin/quarto",
        # User-local install
        str(Path.home() / ".local" / "bin" / "quarto"),
        # Conda env
        str(Path(sys.executable).parent / "quarto"),
    ]
    for c in candidates:
        if Path(c).is_file():
            return c

    return "quarto"  # Fall back to bare name (will raise FileNotFoundError)


def _render_quarto_report(
    matrix_path: str,
    feat_name: str,
    report_dir: Path,
    python_exe: str,
    cvd_safe: bool = False,
    shap_samples: int = 500,
    shap_features: int = 10,
) -> tuple[bool, str]:
    """Render a Quarto HTML dashboard for a single feature matrix."""
    import subprocess
    import os

    pkg_dir = Path(__file__).resolve().parent
    template_dir = pkg_dir / "templates"
    template_file = template_dir / "report_template.qmd"

    if not template_file.exists():
        return False, f"Template not found at {template_file}"

    env = os.environ.copy()
    env["QUARTO_PYTHON"] = str(python_exe)

    quarto_bin = _find_quarto()
    out_html = Path(report_dir) / f"{feat_name}_dashboard.html"
    log_file = Path(report_dir) / f"{feat_name}_render.log"
    cmd = [
        quarto_bin,
        "render",
        "report_template.qmd",
        "--log-level",
        "DEBUG",
        "-P",
        f"matrix_path:{Path(matrix_path).absolute()}",
        "-P",
        f"feature_name:{feat_name}",
        "-P",
        f"cvd_safe:{str(cvd_safe).lower()}",
        "-P",
        f"shap_samples:{shap_samples}",
        "-P",
        f"shap_features:{shap_features}",
        "--output",
        out_html.name,
        "--output-dir",
        str(out_html.parent.absolute()),
    ]

    try:
        subprocess.run(
            cmd,
            env=env,
            capture_output=True,
            text=True,
            check=True,
            timeout=600,
            cwd=str(template_dir),
        )
        return True, str(out_html)
    except subprocess.CalledProcessError as exc:
        # Write full output to disk for debugging — truncated console output hides errors
        with open(log_file, "w") as f:
            f.write(f"=== QUARTO RENDER DEBUG LOG: {feat_name} ===\n")
            f.write(f"CMD: {' '.join(cmd)}\n")
            f.write(f"EXIT CODE: {exc.returncode}\n\n")
            f.write("=== STDOUT ===\n")
            f.write(exc.stdout or "(empty)")
            f.write("\n\n=== STDERR ===\n")
            f.write(exc.stderr or "(empty)")
        # Return concise error + pointer to full log
        err_summary = f"Exit code {exc.returncode}. Full log: {log_file}\n"
        # Extract actual error lines (skip cell progress noise)
        for stream in [exc.stdout, exc.stderr]:
            if not stream:
                continue
            for line in stream.splitlines():
                line_s = line.strip()
                if any(
                    kw in line_s.lower()
                    for kw in [
                        "error",
                        "exception",
                        "traceback",
                        "failed",
                        "fatal",
                        "not found",
                    ]
                ):
                    err_summary += f"  >> {line_s}\n"
        return False, err_summary
    except subprocess.TimeoutExpired:
        return False, "Timeout (>600s)"


@app.command()
def run(
    cancer_samplesheet: Path = typer.Option(...),
    healthy_xs1_samplesheet: Path = typer.Option(...),
    healthy_xs2_samplesheet: Path = typer.Option(...),
    cbioportal_dir: Path = typer.Option(...),
    krewlyzer_dir: list[str] = typer.Option(..., help="krewlyzer output directory"),
    output: Path = typer.Option("output/", help="Output directory"),
    min_vaf: float = typer.Option(0.01),
    min_fragments: int = typer.Option(
        2000, help="Min fragments PF for Depth QC (samples below are Insufficient Data)"
    ),
    min_variants: int = typer.Option(1),
    features: Optional[str] = typer.Option(
        None, help="Comma-separated features to run"
    ),
    tier: Optional[int] = typer.Option(None, help="Run features of this tier only"),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    skip_report: bool = typer.Option(False, help="Skip HTML report generation"),
    cv_folds: int = typer.Option(
        5, "--cv-folds", help="Number of cross-validation folds (3-20, default 5)"
    ),
    impute_strategy: str = typer.Option(
        "median",
        "--impute-strategy",
        help="Imputation strategy for missing values: median, mean, or zero",
    ),
    export_duckdb: bool = typer.Option(
        False,
        "--export-duckdb",
        help="Export a persistent duckdb data lake containing all feature matrices",
    ),
    chunk_size: str = typer.Option(
        "auto",
        "--chunk-size",
        help=(
            "Samples per DuckDB read batch. 'auto' (default) probes parquet "
            "row density at runtime, or pass an integer to override "
            "(e.g. --chunk-size 200)."
        ),
    ),
    top_n: int = typer.Option(
        None,
        "--top-n",
        help="[DEPRECATED] Use --top-percentile instead. If set, overrides --top-percentile with a fixed count.",
    ),
    top_percentile: float = typer.Option(
        10.0,
        "--top-percentile",
        help="Top X%% of features to select per metric (AUC, MI). The union of both sets feeds into models.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards. Lower values reduce memory usage.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP beeswarm/waterfall plots.",
    ),
    resume: bool = typer.Option(
        False,
        "--resume",
        help="Skip models whose AUC results already exist in the output JSON. "
        "Enables incremental runs: first CPU, then GPU with --resume.",
    ),
    compute_univariate_auc: bool = typer.Option(
        True,
        "--compute-univariate-auc",
        help="Compute per-feature univariate LR AUC. Required for hybrid selection (default: True).",
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        "--ch-hotspot-maf",
        help="Optional TSV of CH hotspot variants for CH-only demotion. "
        "Samples with only CH mutations are demoted to Possible ctDNA−.",
    ),
    gpu_models_flag: str = typer.Option(
        "",
        "--gpu-models",
        help="Comma-separated GPU models to run after CPU models: tabpfn,tabicl. "
        "Empty (default) = CPU only. Requires torch + model packages (pip install kreview[gpu]).",
    ),
    no_finetune: bool = typer.Option(
        False,
        "--no-finetune",
        help="Use zero-shot inference for GPU models instead of fine-tuning (not recommended).",
    ),
    finetune_epochs: int = typer.Option(
        30,
        "--finetune-epochs",
        help="Number of fine-tuning epochs for GPU foundation models.",
    ),
    device: str = typer.Option(
        "cuda",
        "--device",
        help="PyTorch device for GPU models: cuda, cpu.",
    ),
):
    """Run full pipeline: label → extract → evaluate → report."""
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler
    from kreview.registry import get_all_evaluators

    from kreview.eval_engine import evaluate_feature, cpu_models
    import glob
    import json

    def _echo(msg):
        print(msg, flush=True)

    if cv_folds < 3 or cv_folds > 20:
        _echo("ERROR: --cv-folds must be between 3 and 20")
        raise typer.Exit(code=1)
    if impute_strategy not in ["median", "mean", "zero"]:
        _echo("ERROR: --impute-strategy must be median, mean, or zero")
        raise typer.Exit(code=1)
    if not (1.0 <= top_percentile <= 100.0):
        _echo("ERROR: --top-percentile must be between 1.0 and 100.0")
        raise typer.Exit(code=1)
    if top_n is not None:
        _echo(
            "WARNING: --top-n is deprecated and will be removed in v0.1.0. "
            "Use --top-percentile instead. Ignoring --top-n in favor of --top-percentile."
        )

    _echo("=== kreview run ===")
    _echo("Configuration:")
    _echo(f"  --cancer-samplesheet : {cancer_samplesheet}")
    _echo(f"  --healthy-xs1       : {healthy_xs1_samplesheet}")
    _echo(f"  --healthy-xs2       : {healthy_xs2_samplesheet}")
    _echo(f"  --cbioportal-dir    : {cbioportal_dir}")
    _echo(f"  --krewlyzer-dir     : {krewlyzer_dir}")
    _echo(f"  --output            : {output}")
    _echo(f"  --min-vaf           : {min_vaf}")
    _echo(f"  --min-variants      : {min_variants}")
    _echo(f"  --features          : {features or 'ALL'}")
    _echo(f"  --tier              : {tier or 'ALL'}")
    _echo(f"  --cv-folds          : {cv_folds}")
    _echo(f"  --top-percentile    : {top_percentile}")
    _echo(f"  --impute-strategy   : {impute_strategy}")
    _echo(f"  --chunk-size        : {chunk_size}")
    _echo(f"  --shap-samples      : {shap_samples}")
    _echo(f"  --shap-features     : {shap_features}")
    _echo(f"  --skip-report       : {skip_report}")
    _echo(f"  --cvd-safe          : {cvd_safe}")
    _echo(f"  --export-duckdb     : {export_duckdb}")
    _echo(f"  --resume            : {resume}")
    _echo(f"  --compute-univariate-auc : {compute_univariate_auc}")
    _echo(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}")
    _echo(f"  --gpu-models        : {gpu_models_flag or 'disabled (CPU only)'}")
    if gpu_models_flag:
        _echo(f"  --device            : {device}")
        _echo(f"  --finetune          : {not no_finetune}")
        _echo(f"  --finetune-epochs   : {finetune_epochs}")
    _echo("")

    # ── Parse GPU models ──
    gpu_model_list = (
        [m.strip() for m in gpu_models_flag.split(",") if m.strip()]
        if gpu_models_flag
        else []
    )
    # All requested models (CPU + GPU) for resume checking
    cpu_model_names = {"lr", "rf", "xgb"}
    all_requested_models = cpu_model_names | set(gpu_model_list)
    if gpu_model_list:
        _echo(f"  GPU models requested: {gpu_model_list}")
        log.info(
            "gpu_models_requested",
            models=gpu_model_list,
            device=device,
            finetune=not no_finetune,
        )

    # ── Validate --chunk-size ──
    # 'auto' passes through to iter_feature_chunks which probes row density.
    # An integer string is converted; anything else is rejected early.
    effective_chunk: int | str = chunk_size
    if chunk_size != "auto":
        try:
            effective_chunk = int(chunk_size)
            if effective_chunk < 1:
                raise ValueError("chunk_size must be positive")
        except ValueError:
            _echo(
                f"ERROR: --chunk-size must be 'auto' or a positive integer, "
                f"got '{chunk_size}'"
            )
            raise typer.Exit(code=1)

    # ── Step 1: Labels ──
    _echo("Step 1: Generating Labels...")
    t0 = time.time()
    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        list(krewlyzer_dir),
    )
    # LabelConfig uses a fixed chunk_size — metadata is always ~1 row/sample.
    config = LabelConfig(
        min_vaf=min_vaf,
        min_fragments=min_fragments,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )
    labeler = CtDNALabeler(paths, config)
    labels_df = labeler.label_all()
    _echo(f"  Labels: {len(labels_df)} samples in {time.time()-t0:.1f}s")
    _echo(f"  Distribution:\n{labels_df['label'].value_counts().to_string()}")

    # ── Step 2: Resolve evaluators ──
    _echo("Step 2: Resolving Feature Evaluators...")
    all_evals = get_all_evaluators()
    target_evals = []
    feat_filter = features.split(",") if features else []
    for e in all_evals:
        if tier is not None and getattr(e, "tier", -1) != tier:
            continue
        if feat_filter and getattr(e, "name", "") not in feat_filter:
            continue
        target_evals.append(e)

    if not target_evals:
        _echo(f"ERROR: No evaluators matched. Available: {[e.name for e in all_evals]}")
        raise typer.Exit(code=1)
    _echo(f"  Matched: {[e.name for e in target_evals]}")

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    all_sample_ids = list(labels_df["SAMPLE_ID"].unique())

    for e in target_evals:
        _echo(f"\n{'='*60}")
        _echo(f"Processing evaluator: {e.name}")

        # ── Resume checkpoint: model-aware skip logic ──
        # Checks which specific models (auc_lr, auc_rf, auc_tabpfn, etc.) are
        # already computed. Only skips the evaluator entirely when ALL requested
        # models have results. Otherwise, tracks which models need to run.
        existing_results = {}  # loaded JSON for merge (populated if resume + file exists)
        models_to_skip = set()  # model names that already have AUC results
        if resume:
            checkpoint = out_path / f"{e.name}_model_results.json"
            if checkpoint.exists():
                try:
                    with open(checkpoint) as _f:
                        existing_results = json.load(_f)
                    models_to_skip = {
                        k.replace("auc_", "")
                        for k in existing_results
                        if k.startswith("auc_")
                        and isinstance(existing_results[k], (int, float))
                    }
                    remaining = all_requested_models - models_to_skip
                    if not remaining:
                        _echo(
                            f"  SKIP (--resume): {checkpoint.name} — all "
                            f"{len(models_to_skip)} models computed: {sorted(models_to_skip)}"
                        )
                        continue
                    _echo(
                        f"  RESUME: {e.name} — existing: {sorted(models_to_skip)}, "
                        f"remaining: {sorted(remaining)}"
                    )
                    log.info(
                        "resume_partial",
                        evaluator=e.name,
                        existing=sorted(models_to_skip),
                        remaining=sorted(remaining),
                    )
                except Exception as exc:
                    log.warning(
                        "resume_checkpoint_read_failed",
                        evaluator=e.name,
                        error=str(exc),
                    )
                    _echo(
                        f"  WARNING: Could not read {checkpoint.name}: {exc}. "
                        f"Re-running all models."
                    )

        # ── Step 3: Extract features ──
        # Delegates to _extract_evaluator() which handles both SQL pushdown
        # (Path A) and chunked Python streaming (Path B).
        merged = _extract_evaluator(
            e,
            paths,
            all_sample_ids,
            effective_chunk,
            out_path,
            labels_df,
            _echo=_echo,
        )
        if merged is None:
            continue

        # ── Step 4: Statistical Evaluation ──
        _echo(f"Step 4: Running statistical evaluation for '{e.name}'...")
        t3 = time.time()

        # Identify numeric feature columns (exclude label/metadata cols).
        # Uses the canonical LABEL_META_COLS set from core — single source of truth.
        from kreview.core import LABEL_META_COLS
        from kreview.selection import score_features, select_features, build_binary_target

        numeric_cols = [
            c
            for c in merged.select_dtypes(include=np.number).columns
            if c not in LABEL_META_COLS
        ]
        _echo(f"  Found {len(numeric_cols)} numeric feature columns")

        # ── Score all features (univariate AUC + MI + KW/Cohen's d) ──
        try:
            eval_df = score_features(
                merged,
                cv_folds=cv_folds,
                compute_auc=compute_univariate_auc,
            )
        except ValueError as exc:
            _echo(f"  SKIP scoring for {e.name}: {exc}")
            log.warning("score_features_skip", evaluator=e.name, reason=str(exc))
            continue

        eval_out = out_path / f"{e.name}_eval_stats.parquet"
        eval_df.to_parquet(eval_out, index=False)
        _echo(f"  Eval stats: {eval_df.shape[0]} features -> {eval_out}")

        # ── Feature selection (hybrid union: top N% AUC ∪ top N% MI) ──
        try:
            selected_df, selection_qc = select_features(
                merged,
                eval_df,
                top_percentile=top_percentile,
                impute_strategy=impute_strategy,
            )
        except ValueError as exc:
            _echo(f"  SKIP selection for {e.name}: {exc}")
            log.warning("select_features_skip", evaluator=e.name, reason=str(exc))
            continue

        # Save selected matrix — overwrites the full matrix from extract
        # so downstream report/DuckDB export use the reduced feature set.
        matrix_out = out_path / f"{e.name}_matrix.parquet"
        selected_df.to_parquet(matrix_out, index=False)
        _echo(f"  Selected matrix: {selected_df.shape[1]} cols -> {matrix_out}")

        # Save selection QC
        qc_out = out_path / f"{e.name}_selection_qc.json"
        with open(qc_out, "w") as _f:
            json.dump(selection_qc, _f, indent=2, default=str)

        # Determine selected feature columns from the result
        top_feats = [
            c for c in selected_df.columns if c not in LABEL_META_COLS
        ]
        _echo(
            f"  Feature Selection (top {top_percentile}%): "
            f"{len(top_feats)}/{len(numeric_cols)} features "
            f"(AUC\u2229MI={selection_qc.get('n_overlap_both', 0)}, "
            f"AUC-only={selection_qc.get('n_auc_only', 0)}, "
            f"MI-only={selection_qc.get('n_mi_only', 0)})"
        )

        # ── Prepare model inputs ──
        # Build binary target from the selected matrix using the shared function.
        try:
            model_df, y = build_binary_target(selected_df)
        except ValueError as exc:
            _echo(f"  SKIP model for {e.name}: {exc}")
            log.warning("build_target_skip", evaluator=e.name, reason=str(exc))
            continue

        import warnings

        warnings.filterwarnings("ignore", message=".*use_label_encoder.*")

        _echo(f"  Imputation: {impute_strategy}")

        X = _impute(model_df[top_feats], impute_strategy).values
        c_types = model_df.get("CANCER_TYPE", None)
        if c_types is not None:
            c_types = c_types.values
        a_types = model_df.get("access_version", None)
        if a_types is not None:
            a_types = a_types.values

        def _fmt(v):
            return "N/A" if v is None else f"{v:.3f}"

        model_out = out_path / f"{e.name}_model_results.json"

        # ── CPU models (lr, rf, xgb) ──
        skip_cpu = cpu_model_names.issubset(models_to_skip)
        if skip_cpu:
            _echo("  CPU models: SKIP (--resume, already computed)")
            model_res = existing_results  # use cached results
            lr_model, rf_model, xgb_model = None, None, None
        else:
            model_res, lr_model, rf_model, xgb_model = cpu_models(
                X,
                y,
                feature_names=top_feats,
                cancer_types=c_types,
                assays=a_types,
                n_folds=cv_folds,
            )
            _echo(
                f"  CPU: AUC_LR={_fmt(model_res.get('auc_lr'))}, "
                f"AUC_RF={_fmt(model_res.get('auc_rf'))}, "
                f"AUC_XGB={_fmt(model_res.get('auc_xgb'))}"
            )

        # ── GPU models (tabpfn, tabicl) ──
        if gpu_model_list:
            gpu_remaining = [
                m for m in gpu_model_list if m not in models_to_skip
            ]
            if not gpu_remaining:
                _echo("  GPU models: SKIP (--resume, already computed)")
            else:
                _echo(
                    f"  Running GPU models: {gpu_remaining} "
                    f"(device={device}, finetune={not no_finetune})"
                )
                try:
                    from kreview.eval_engine import gpu_models

                    gpu_res, _ = gpu_models(
                        X, y,
                        feature_names=top_feats,
                        cancer_types=c_types,
                        assays=a_types,
                        n_folds=cv_folds,
                        models=tuple(gpu_remaining),
                        device=device,
                        finetune=not no_finetune,
                        finetune_epochs=finetune_epochs,
                        compute_shap=False,
                    )
                    model_res.update(gpu_res)
                    for mn in gpu_remaining:
                        _echo(f"  GPU/{mn}: AUC={_fmt(model_res.get(f'auc_{mn}'))}")
                    log.info(
                        "gpu_models_complete",
                        evaluator=e.name,
                        models=gpu_remaining,
                        aucs={mn: model_res.get(f"auc_{mn}") for mn in gpu_remaining},
                    )
                except ImportError as exc:
                    log.error("gpu_import_failed", evaluator=e.name, error=str(exc), hint="pip install kreview[gpu]")
                    _echo(f"  ERROR: GPU dependencies not available: {exc}. Install with: pip install kreview[gpu]")
                except Exception as exc:
                    log.error("gpu_models_failed", evaluator=e.name, error=str(exc))
                    _echo(f"  ERROR: GPU models failed: {exc}")

        # ── Enrich results + save ──
        if "error" not in model_res:
            model_res["evaluator"] = e.name
            model_res["top_features"] = top_feats
            model_res["selection_qc"] = selection_qc
            if "sample_id" in model_df.columns:
                model_res["oof_sample_ids"] = model_df["sample_id"].tolist()
            elif "SAMPLE_ID" in model_df.columns:
                model_res["oof_sample_ids"] = model_df["SAMPLE_ID"].tolist()
            all_aucs = {
                k: v for k, v in model_res.items()
                if k.startswith("auc_") and isinstance(v, (int, float))
                and not k.endswith("_ci_lower") and not k.endswith("_ci_upper")
            }
            if all_aucs:
                model_res["best_model"] = max(all_aucs, key=all_aucs.get)
                model_res["best_auc"] = all_aucs[model_res["best_model"]]

        if existing_results and existing_results is not model_res:
            existing_results.update(model_res)
            model_res = existing_results

        import joblib

        if lr_model is not None:
            joblib.dump(lr_model, out_path / f"{e.name}_lr_model.joblib")
        if rf_model is not None:
            joblib.dump(rf_model, out_path / f"{e.name}_rf_model.joblib")
        if xgb_model is not None:
            joblib.dump(xgb_model, out_path / f"{e.name}_xgb_model.joblib")

        with open(model_out, "w") as f:
            json.dump(model_res, f, indent=2, default=str)

        _echo(f"  Output: {model_out}")

        eval_sec = time.time() - t3
        _echo(f"  Evaluation complete in {eval_sec:.1f}s")

    # ── Step 4a: Fuse per-evaluator matrices into super-matrix ──
    _echo("\nStep 4a: Fusing per-evaluator matrices...")
    super_matrix_path = None
    try:
        from kreview.core import fuse_matrices

        fused_df = fuse_matrices(out_path)
        if fused_df.empty:
            _echo("  SKIP fuse: no matrices found or all samples filtered.")
            log.warning("fuse_skip_empty", output_dir=str(out_path))
        else:
            super_matrix_path = out_path / "super_matrix.parquet"
            n_features = len([c for c in fused_df.columns if "__" in c])
            _echo(
                f"  Fused: {len(fused_df)} samples x {n_features} features "
                f"-> {super_matrix_path}"
            )
            log.info(
                "fuse_complete",
                n_samples=len(fused_df),
                n_features=n_features,
                output=str(super_matrix_path),
            )
    except Exception as exc:
        _echo(f"  ERROR: Fuse failed: {exc}")
        log.error("fuse_failed", error=str(exc))

    # ── Step 4b: Cross-evaluator multimodal evaluation ──
    multimodal_out = out_path / "multimodal_results.json"
    if super_matrix_path is not None and super_matrix_path.exists():
        _echo("\nStep 4b: Running multimodal evaluation (stacking + ablation)...")
        try:
            from kreview.eval_engine import multimodal_eval

            mm_results = multimodal_eval(
                results_dir=out_path,
                super_matrix_path=super_matrix_path,
                n_folds=cv_folds,
            )
            with open(multimodal_out, "w") as f:
                json.dump(mm_results, f, indent=2, default=str)

            # Display summary
            best_single = mm_results.get("best_single_evaluator", "?")
            best_single_auc = mm_results.get("best_single_auc", 0)
            _echo(f"  Best single evaluator: {best_single} (AUC={best_single_auc:.3f})")

            stacking = mm_results.get("stacking", {})
            for k, v in sorted(stacking.items()):
                if k.startswith("auc_stacking_") and not k.endswith(("_ci_lower", "_ci_upper")):
                    mn = k.replace("auc_stacking_", "")
                    _echo(f"  Stacking/{mn}: AUC={v:.3f}")

            _echo(f"  Output: {multimodal_out}")
            log.info(
                "multimodal_complete",
                best_single=best_single,
                best_single_auc=best_single_auc,
                n_evaluators=mm_results.get("n_evaluators", 0),
            )
        except Exception as exc:
            _echo(f"  ERROR: Multimodal evaluation failed: {exc}")
            log.error("multimodal_failed", error=str(exc))
    else:
        _echo("\nStep 4b: SKIP multimodal — no super-matrix available.")
        log.info("multimodal_skip", reason="no super-matrix")
    # ── Step 4c: Build Cross-Evaluator Scoreboard ──
    # Check if any evaluators produced model results
    model_jsons = list(out_path.glob("*_model_results.json"))
    if not model_jsons:
        _echo("\n  WARNING: No evaluators produced model results — skipping scoreboard and reports.")
        log.warning("no_evaluator_results", output_dir=str(out_path))
    else:
        _echo(f"\n  {len(model_jsons)} evaluator(s) produced model results.")

    from kreview.scoreboard import build_scoreboard

    sb = build_scoreboard(out_path)
    if not sb.empty:
        sb.to_parquet(out_path / "scoreboard_combined__all.parquet")
        sb.to_csv(out_path / "scoreboard_combined__all.csv", index=False)
        _echo(f"\n  Scoreboard: {len(sb)} evaluators ranked")
        _echo("  Top 5 by AUC:")
        for _, row in sb.head(5).iterrows():
            best = row["best_auc"]
            auc_str = f"{best:.3f}" if not pd.isna(best) else "N/A"
            _echo(f"    {row['evaluator']:<30} AUC={auc_str}")
        _echo("  -> scoreboard_combined__all.parquet")

    # ── Step 5: Generate HTML Dashboards ──
    if not skip_report:
        report_dir = out_path / "reports"
        report_dir.mkdir(parents=True, exist_ok=True)

        matrices = glob.glob(str(out_path / "*_matrix.parquet"))
        if matrices:
            _echo(f"\nStep 5: Generating {len(matrices)} HTML Dashboard(s)...")
            for p in matrices:
                feat_name = Path(p).name.replace("_matrix.parquet", "")
                _echo(f"  Rendering {feat_name}...")
                ok, msg = _render_quarto_report(
                    p,
                    feat_name,
                    report_dir,
                    sys.executable,
                    cvd_safe=cvd_safe,
                    shap_samples=shap_samples,
                    shap_features=shap_features,
                )
                if ok:
                    _echo(f"  Saved: {msg}")
                else:
                    _echo(f"  Quarto FAILED for {feat_name}: {msg}")

    # ── Step 6: DuckLake Export ──
    if export_duckdb:
        _echo("\nStep 6: Exporting Unified DuckLake...")
        import duckdb

        db_path = out_path / "kreview_lake.duckdb"
        if db_path.exists():
            db_path.unlink()
        try:
            con = duckdb.connect(str(db_path))
            matrices = glob.glob(str(out_path / "*_matrix.parquet"))
            for p in matrices:
                table_name = (
                    Path(p).name.replace("_matrix.parquet", "").replace(".", "_")
                )
                _echo(f"  Importing {table_name} into DuckLake...")
                con.execute(
                    f"CREATE TABLE {table_name} AS SELECT * FROM read_parquet('{p}')"
                )
            con.close()
            _echo(f"  DuckLake saved securely directly to: {db_path}")
        except Exception as e:
            log.error("ducklake_export_failed", error=str(e), db_path=str(db_path))
            _echo(f"  DuckLake creation failed: {e}")

    total_sec = time.time() - t0
    _echo(f"\n=== Workflow complete in {total_sec:.1f}s ===")


In [ ]:
# | export
@app.command()
def report(
    input_dir: Path = typer.Option(..., help="Directory with *_matrix.parquet files"),
    out_dir: Path = typer.Option(
        "reports/", help="Directory to deposit Quarto reports"
    ),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP plots.",
    ),
):
    """Re-generate HTML Dashboards from existing matrix parquet files.

    Scans ``input_dir`` for ``*_matrix.parquet`` files, renders each as a
    standalone Quarto HTML dashboard, and writes them to ``out_dir``.
    Each evaluator is rendered independently so a single failure does not
    block the remaining dashboards.
    """
    import glob
    import sys
    import time as _time

    import structlog

    _log = structlog.get_logger()

    print("=== kreview report ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --input-dir     : {input_dir}", flush=True)
    print(f"  --out-dir       : {out_dir}", flush=True)
    print(f"  --cvd-safe      : {cvd_safe}", flush=True)
    print(f"  --shap-samples  : {shap_samples}", flush=True)
    print(f"  --shap-features : {shap_features}", flush=True)
    print("", flush=True)

    in_path = Path(input_dir).absolute()
    matrices = sorted(glob.glob(str(in_path / "*_matrix.parquet")))
    if not matrices:
        _log.warning("report_no_matrices", input_dir=str(in_path))
        print(f"No *_matrix.parquet files found in {in_path}", flush=True)
        return

    _log.info("report_started", input_dir=str(in_path), n_matrices=len(matrices))

    out_path = Path(out_dir).absolute()
    out_path.mkdir(parents=True, exist_ok=True)

    succeeded = 0
    failed = 0
    failed_names = []

    for p in matrices:
        feat_name = Path(p).name.replace("_matrix.parquet", "")
        print(f"Rendering dashboard for {feat_name}...", flush=True)
        t_start = _time.perf_counter()

        # S-01: Per-evaluator try/except — one failure does not block others
        try:
            ok, msg = _render_quarto_report(
                p,
                feat_name,
                out_path,
                sys.executable,
                cvd_safe=cvd_safe,
                shap_samples=shap_samples,
                shap_features=shap_features,
            )
            elapsed = _time.perf_counter() - t_start
            if ok:
                print(f"  Saved: {msg} ({elapsed:.1f}s)", flush=True)
                _log.info(
                    "report_rendered",
                    evaluator=feat_name,
                    seconds=f"{elapsed:.1f}",
                    output=msg,
                )
                succeeded += 1
            else:
                print(f"  FAILED: {msg} ({elapsed:.1f}s)", flush=True)
                _log.error(
                    "report_render_failed",
                    evaluator=feat_name,
                    seconds=f"{elapsed:.1f}",
                    error=msg[:500],
                )
                failed += 1
                failed_names.append(feat_name)
        except Exception as exc:
            elapsed = _time.perf_counter() - t_start
            print(
                f"  EXCEPTION rendering {feat_name}: {exc} ({elapsed:.1f}s)",
                flush=True,
            )
            _log.error(
                "report_render_exception",
                evaluator=feat_name,
                seconds=f"{elapsed:.1f}",
                error=str(exc),
            )
            failed += 1
            failed_names.append(feat_name)

    # L-03: Summary with pass/fail counts
    total = len(matrices)
    print(
        f"\nReport summary: {succeeded}/{total} succeeded, {failed}/{total} failed",
        flush=True,
    )
    if failed_names:
        print(f"  Failed evaluators: {', '.join(failed_names)}", flush=True)
    _log.info(
        "report_completed",
        succeeded=succeeded,
        failed=failed,
        total=total,
        failed_evaluators=failed_names,
    )


In [ ]:
# | test
# Smoke test
assert hasattr(app, "registered_commands")

In [ ]:
#| export
@app.command()
def extract(
    cancer_samplesheet: Path = typer.Option(..., help="Cancer samplesheet CSV"),
    healthy_xs1_samplesheet: Path = typer.Option(
        ..., help="Healthy XS1 samplesheet CSV"
    ),
    healthy_xs2_samplesheet: Path = typer.Option(
        ..., help="Healthy XS2 samplesheet CSV"
    ),
    cbioportal_dir: Path = typer.Option(..., help="Directory with cBioPortal files"),
    krewlyzer_dir: list[str] = typer.Option(..., help="krewlyzer output directory"),
    output: Path = typer.Option("output/", help="Output directory for matrices"),
    min_vaf: float = typer.Option(
        0.01, help="Min VAF for Possible ctDNA+ (default 1%)"
    ),
    min_fragments: int = typer.Option(
        2000, help="Min fragments PF for Depth QC (samples below are Insufficient Data)"
    ),
    min_variants: int = typer.Option(
        1, help="Min # variants passing VAF for Possible ctDNA+"
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        help="Optional TSV of CH hotspot variants for CH-only demotion.",
    ),
    features: Optional[str] = typer.Option(
        None, help="Comma-separated evaluator names (default: all)"
    ),
    tier: Optional[int] = typer.Option(None, help="Run only this tier"),
    chunk_size: str = typer.Option(
        "auto",
        "--chunk-size",
        help=(
            "Samples per DuckDB read batch. 'auto' (default) probes parquet "
            "row density at runtime, or pass an integer to override "
            "(e.g. --chunk-size 200)."
        ),
    ),
):
    """Label samples and extract feature matrices (no eval/model/report).

    Runs the labeling pipeline, then extracts features for each matched
    evaluator into ``*_matrix.parquet`` files. This is the first half of
    ``kreview run``, designed for parallelized Nextflow execution.
    """
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler
    from kreview.registry import get_all_evaluators

    print("=== kreview extract ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --cancer-samplesheet : {cancer_samplesheet}", flush=True)
    print(f"  --healthy-xs1       : {healthy_xs1_samplesheet}", flush=True)
    print(f"  --healthy-xs2       : {healthy_xs2_samplesheet}", flush=True)
    print(f"  --cbioportal-dir    : {cbioportal_dir}", flush=True)
    print(f"  --krewlyzer-dir     : {krewlyzer_dir}", flush=True)
    print(f"  --output            : {output}", flush=True)
    print(f"  --features          : {features or 'all'}", flush=True)
    print(f"  --tier              : {tier or 'all'}", flush=True)
    print(f"  --chunk-size        : {chunk_size}", flush=True)
    print(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}", flush=True)
    print("", flush=True)

    t0 = time.time()

    # ── Step 1: Label ──
    print("Step 1: Labeling...", flush=True)
    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        krewlyzer_dir,
    )
    config = LabelConfig(
        min_vaf=min_vaf,
        min_fragments=min_fragments,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )
    labeler = CtDNALabeler(paths, config)
    labels_df = labeler.label_all()
    print(f"  Labels: {len(labels_df)} samples in {time.time()-t0:.1f}s", flush=True)
    print(
        f"  Distribution:\n{labels_df['label'].value_counts().to_string()}", flush=True
    )

    # ── Step 2: Resolve evaluators ──
    print("Step 2: Resolving Feature Evaluators...", flush=True)
    all_evals = get_all_evaluators()
    target_evals = []
    feat_filter = features.split(",") if features else []
    for e in all_evals:
        if tier is not None and getattr(e, "tier", -1) != tier:
            continue
        if feat_filter and getattr(e, "name", "") not in feat_filter:
            continue
        target_evals.append(e)

    if not target_evals:
        print(
            f"ERROR: No evaluators matched. Available: {[e.name for e in all_evals]}",
            flush=True,
        )
        raise typer.Exit(code=1)
    print(f"  Matched: {[e.name for e in target_evals]}", flush=True)

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    all_sample_ids = list(labels_df["SAMPLE_ID"].unique())

    # Parse chunk_size: 'auto' passes through to iter_feature_chunks,
    # integer string is converted, anything else is rejected.
    effective_chunk: int | str = chunk_size
    if chunk_size != "auto":
        try:
            effective_chunk = int(chunk_size)
            if effective_chunk < 1:
                raise ValueError("chunk_size must be positive")
        except ValueError:
            print(
                f"ERROR: --chunk-size must be 'auto' or a positive integer, "
                f"got '{chunk_size}'",
                flush=True,
            )
            raise typer.Exit(code=1)
    print(f"  Effective chunk size: {effective_chunk}", flush=True)

    # ── Step 3: Extract each evaluator ──
    extracted = 0
    for e in target_evals:
        print(f"\n{'='*60}", flush=True)
        print(f"Extracting evaluator: {e.name}", flush=True)

        merged = _extract_evaluator(
            e,
            paths,
            all_sample_ids,
            effective_chunk,
            out_path,
            labels_df,
        )
        if merged is not None:
            extracted += 1

    elapsed = time.time() - t0
    print(
        f"\n=== Extract complete: {extracted}/{len(target_evals)} evaluators "
        f"in {elapsed:.1f}s ===",
        flush=True,
    )
    log.info(
        "extract_complete",
        n_extracted=extracted,
        n_total=len(target_evals),
        elapsed_sec=round(elapsed, 1),
    )


In [ ]:
#| export
@app.command()
def fuse(
    output_dir: Path = typer.Option(
        ..., "--output-dir", help="Directory containing *_matrix.parquet files"
    ),
    min_evaluators: int = typer.Option(
        1,
        "--min-evaluators",
        help="Minimum number of evaluators a sample must appear in to be retained",
    ),
    output_name: str = typer.Option(
        "super_matrix.parquet",
        "--output-name",
        help="Filename for the fused super-matrix (written to --output-dir)",
    ),
):
    """Fuse per-evaluator matrices into a single super-matrix.

    Discovers all ``*_matrix.parquet`` files in ``--output-dir``, extracts
    their feature columns (prefixed with evaluator name), outer-joins on
    SAMPLE_ID, and writes ``super_matrix.parquet`` for downstream multimodal
    evaluation.
    """
    import time as _time

    from kreview.core import fuse_matrices

    print("=== kreview fuse ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --output-dir     : {output_dir}", flush=True)
    print(f"  --min-evaluators : {min_evaluators}", flush=True)
    print(f"  --output-name    : {output_name}", flush=True)
    print("", flush=True)

    if min_evaluators < 1:
        print("ERROR: --min-evaluators must be >= 1", flush=True)
        raise typer.Exit(code=1)

    t0 = _time.time()
    result = fuse_matrices(
        output_dir,
        min_evaluators=min_evaluators,
        output_name=output_name,
    )

    if result.empty:
        print(
            "ERROR: No matrices found or all samples filtered. Nothing to fuse.",
            flush=True,
        )
        raise typer.Exit(code=1)

    n_features = len([c for c in result.columns if "__" in c])
    elapsed = _time.time() - t0
    print(
        f"  Fused: {len(result)} samples x {n_features} features "
        f"from {result.get('n_evaluators', pd.Series()).max() or '?'} evaluators "
        f"in {elapsed:.1f}s",
        flush=True,
    )
    print(f"  Output: {output_dir / output_name}", flush=True)